In [5]:
import tensorflow as tf
print(tf.__version__)


2.20.0


In [6]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ========= PATHS =========
BASE_DIR = r"C:\Users\user\Desktop\Skin Detector\archive"
DATASET_DIR = os.path.join(BASE_DIR, "dataset")            # your dataset folder
SPLIT_DIR   = os.path.join(BASE_DIR, "dataset_split")      # will be created
MODELS_DIR  = os.path.join(BASE_DIR, "models")             # model save here

os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("BASE_DIR  :", BASE_DIR)
print("DATASET   :", DATASET_DIR)
print("SPLIT_DIR :", SPLIT_DIR)
print("MODELS_DIR:", MODELS_DIR)

print("\nDataset folder exists? ", os.path.exists(DATASET_DIR))
print("Split folder exists?    ", os.path.exists(SPLIT_DIR))
print("Models folder exists?   ", os.path.exists(MODELS_DIR))


BASE_DIR  : C:\Users\user\Desktop\Skin Detector\archive
DATASET   : C:\Users\user\Desktop\Skin Detector\archive\dataset
SPLIT_DIR : C:\Users\user\Desktop\Skin Detector\archive\dataset_split
MODELS_DIR: C:\Users\user\Desktop\Skin Detector\archive\models

Dataset folder exists?  True
Split folder exists?     True
Models folder exists?    True


In [7]:
classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print("Total folders (classes) found:", len(classes))
print("Classes:", classes)


Total folders (classes) found: 3
Classes: ['test', 'train', 'val']


In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

IMG_SIZE = 224
BATCH = 16

SPLIT_DIR = r"C:\Users\user\Desktop\Skin Detector\archive\dataset"

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True
)

val_test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    os.path.join(SPLIT_DIR, "train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical"
)

val_data = val_test_gen.flow_from_directory(
    os.path.join(SPLIT_DIR, "val"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical"
)

test_data = val_test_gen.flow_from_directory(
    os.path.join(SPLIT_DIR, "test"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical",
    shuffle=False
)


Found 1161 images belonging to 6 classes.
Found 330 images belonging to 6 classes.
Found 338 images belonging to 6 classes.


In [9]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model

NUM_CLASSES = train_data.num_classes
print("NUM_CLASSES =", NUM_CLASSES)

base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False  # start with frozen base

x = GlobalAveragePooling2D()(base.output)
x = Dropout(0.3)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


NUM_CLASSES = 6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 112, 112, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 112, 112, 32)      │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 112, 112, 32)      │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 32)      │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 112, 112, 32)      │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 112, 112, 32)      │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 112, 112, 96)      │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 112, 112, 96)      │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 112, 112, 96)      │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 113, 113, 96)      │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 56, 56, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,587,462 (9.87 MB)

 Trainable params: 329,478 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
import os
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# ===============================
# Model save directory
# ===============================
MODEL_DIR = r"C:\Users\user\Desktop\Skin Detector\archive\models"
os.makedirs(MODEL_DIR, exist_ok=True)

# ===============================
# Callbacks
# ===============================

# 1. Save BEST model in KERAS format (.keras)
checkpoint = ModelCheckpoint(
    filepath=os.path.join(MODEL_DIR, "skin_detector_best_7cls.keras"),
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# 2. Early stopping (overfitting protection)
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# 3. Reduce learning rate if model stuck
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# ===============================
# Train model
# ===============================
EPOCHS = 25

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop, reduce_lr]
)


Epoch 1/25
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7271 - loss: 0.7183  
Epoch 1: val_accuracy improved from None to 0.77576, saving model to C:\Users\user\Desktop\Skin Detector\archive\models\skin_detector_best_7cls.keras
73/73 ━━━━━━━━━━━━━━━━━━━━ 113s 2s/step - accuracy: 0.7416 - loss: 0.6826 - val_accuracy: 0.7758 - val_loss: 0.6239 - learning_rate: 9.0000e-06
Epoch 2/25
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7419 - loss: 0.7017  
Epoch 2: val_accuracy improved from 0.77576 to 0.77879, saving model to C:\Users\user\Desktop\Skin Detector\archive\models\skin_detector_best_7cls.keras
73/73 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - accuracy: 0.7399 - loss: 0.6856 - val_accuracy: 0.7788 - val_loss: 0.6255 - learning_rate: 9.0000e-06
Epoch 3/25
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7304 - loss: 0.6791  
Epoch 3: val_accuracy did not improve from 0.77879
73/73 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.7347 - loss: 0.6861 - val_accuracy: 0.7758 - val_

In [ ]:
import matplotlib.pyplot as plt

# Accuracy
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Evaluate on test data
test_loss, test_acc = model.evaluate(test_data)
print(f"\n✅ Test Accuracy: {test_acc*100:.2f}%")

# Predictions
y_pred = model.predict(test_data)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_data.classes

# Class names
class_names = list(test_data.class_indices.keys())

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred_classes, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
import os

BASE_DIR = r"C:\Users\user\Desktop\Skin Detector\archive\dataset"

diseases = ["Acne", "Eczema", "Psoriasis"]

def count_images(folder_path):
    if not os.path.exists(folder_path):
        return 0
    return len([
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

print("📊 DATASET IMAGE COUNT (Train / Val / Test)\n")

for disease in diseases:
    train_path = os.path.join(BASE_DIR, "train", disease)
    val_path   = os.path.join(BASE_DIR, "val", disease)
    test_path  = os.path.join(BASE_DIR, "test", disease)

    train_count = count_images(train_path)
    val_count   = count_images(val_path)
    test_count  = count_images(test_path)

    total = train_count + val_count + test_count

    print(f"🦠 {disease}")
    print(f"  Train: {train_count}")
    print(f"  Val  : {val_count}")
    print(f"  Test : {test_count}")
    print(f"  Total: {total}")
    print("-" * 35)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers


In [ ]:
# Dataset paths
BASE_DIR = r"C:\Users\user\Desktop\Skin Detector\archive\dataset"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

# Image & training settings
IMG_SIZE = 224
BATCH = 16
NUM_CLASSES = 7


In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.25,
    shear_range=0.2,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True
)

val_test_gen = ImageDataGenerator(rescale=1./255)


In [ ]:
train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical"
)

val_data = val_test_gen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical"
)

test_data = val_test_gen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical",
    shuffle=False
)


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # freeze backbone

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

MODEL_DIR = r"C:\Users\user\Desktop\Skin Detector\archive\models"
os.makedirs(MODEL_DIR, exist_ok=True)

checkpoint = ModelCheckpoint(
    os.path.join(MODEL_DIR, "skin_detector_v2_aug.keras"),
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=[checkpoint, early_stop, reduce_lr]
)


In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Val")
plt.legend()
plt.title("Accuracy")

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.legend()
plt.title("Loss")

plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = test_data.classes
y_pred = np.argmax(model.predict(test_data), axis=1)

print(classification_report(y_true, y_pred, target_names=train_data.class_indices.keys()))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=train_data.class_indices.keys(),
            yticklabels=train_data.class_indices.keys())
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# Unfreeze top layers of base model
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history_ft = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[checkpoint, early_stop, reduce_lr],
    class_weight=class_weights
)


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Get class indices from training generator
class_indices = train_data.class_indices
print("Class indices:", class_indices)

# Get labels
y_train = train_data.classes

# Compute class weights
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dictionary
class_weights = dict(enumerate(weights))

print("Class weights:", class_weights)


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history_ft = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[checkpoint, early_stop, reduce_lr],
    class_weight=class_weights
)


In [ ]:
import tensorflow as tf

MODEL_PATH = r"C:\Users\user\Desktop\Skin Detector\archive\models\skin_detector_best_7cls.keras"

best_model = tf.keras.models.load_model(MODEL_PATH)

test_loss, test_acc = best_model.evaluate(test_data)
print("Test Accuracy:", test_acc)


In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# True labels
y_true = test_data.classes

# Model predictions
y_pred_probs = best_model.predict(test_data)
y_pred = np.argmax(y_pred_probs, axis=1)

class_names = list(test_data.class_indices.keys())

print("📊 CLASSIFICATION REPORT\n")
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
# STEP 10: Save final trained model

FINAL_MODEL_PATH = r"C:\Users\user\Desktop\Skin Detector\archive\models\skin_detector_final.keras"

model.save(FINAL_MODEL_PATH)

print("✅ Model successfully saved at:")
print(FINAL_MODEL_PATH)


In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Load trained model
MODEL_PATH = r"C:\Users\user\Desktop\Skin Detector\archive\models\skin_detector_final.keras"
model = tf.keras.models.load_model(MODEL_PATH)

# Class names (same order as training)
class_names = [
    "Acne",
    "Eczema",
    "Normal",
    "Psoriasis",
    "Rosacea",
    "Vitiligo",
    "Warts"
]


In [ ]:
# Image path (change this)
IMG_PATH = r"C:\Users\user\Downloads\WhatsApp Image 2025-12-13 at 2.53.27 PM.jpeg"

IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Resize & normalize
img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)


In [ ]:
pred = model.predict(img_array)
predicted_class = class_names[np.argmax(pred)]
confidence = np.max(pred) * 100

plt.imshow(img)
plt.axis("off")
plt.title(f"Prediction: {predicted_class} ({confidence:.2f}%)")
plt.show()

print("🩺 Predicted Disease:", predicted_class)
print("📊 Confidence:", round(confidence, 2), "%")


In [ ]:
# Image path (change this)
IMG_PATH = r"C:\Users\user\Downloads\WhatsApp Image 2025-12-13 at 10.38.13 PM.jpeg"

IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Resize & normalize
img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)


In [ ]:
pred = model.predict(img_array)
predicted_class = class_names[np.argmax(pred)]
confidence = np.max(pred) * 100

plt.imshow(img)
plt.axis("off")
plt.title(f"Prediction: {predicted_class} ({confidence:.2f}%)")
plt.show()

print("🩺 Predicted Disease:", predicted_class)
print("📊 Confidence:", round(confidence, 2), "%")

In [ ]:
# Image path (change this)
IMG_PATH = r"C:\Users\user\Downloads\WhatsApp Image 2025-12-13 at 11.03.21 PM.jpeg"

IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Resize & normalize
img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

In [ ]:
pred = model.predict(img_array)
predicted_class = class_names[np.argmax(pred)]
confidence = np.max(pred) * 100

plt.imshow(img)
plt.axis("off")
plt.title(f"Prediction: {predicted_class} ({confidence:.2f}%)")
plt.show()

print("🩺 Predicted Disease:", predicted_class)
print("📊 Confidence:", round(confidence, 2), "%")

In [ ]:
# ===============================
# STEP B: Image Upload & Prediction (REPEAT THIS)
# ===============================

IMG_PATH = r"C:\Users\user\Downloads\WhatsApp Image 2025-12-13 at 11.03.21 PM.jpeg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
predicted_index = np.argmax(pred)
confidence = pred[0][predicted_index] * 100
predicted_class = class_names[predicted_index]

# Show Image + Prediction
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({confidence:.2f}%)")
plt.show()

# Print details
print("🧬 Predicted Disease:", predicted_class)
print(f"📊 Confidence: {confidence:.2f} %")

print("\n💊 Recommended Creams / Treatment:")
for cream in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", cream)


In [ ]:
# ===============================
# STEP B: Image Upload & Prediction (REPEAT THIS)
# ===============================

IMG_PATH = r"C:\Users\user\Downloads\WhatsApp Image 2025-12-13 at 2.53.27 PM.jpeg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
predicted_index = np.argmax(pred)
confidence = pred[0][predicted_index] * 100
predicted_class = class_names[predicted_index]

# Show Image + Prediction
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({confidence:.2f}%)")
plt.show()

# Print details
print("🧬 Predicted Disease:", predicted_class)
print(f"📊 Confidence: {confidence:.2f} %")

print("\n💊 Recommended Creams / Treatment:")
for cream in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", cream)


In [16]:
import tensorflow as tf
import numpy as np
from PIL import Image
import io

from ipywidgets import FileUpload, Button, VBox, Output
from IPython.display import display


In [ ]:
model = tf.keras.models.load_model(
    r"C:\Users\user\Desktop\Skin Detector\archive\models\skin_detector_final.keras"
)

class_names = [
    "Acne","Eczema","Normal","Psoriasis",
    "Rosacea","Vitiligo","Warts"
]

recommendations = {
    "Acne": ["Benzoyl Peroxide", "Salicylic Acid"],
    "Eczema": ["Hydrocortisone Cream", "Moisturizer"],
    "Psoriasis": ["Coal Tar", "Vitamin D Cream"],
    "Rosacea": ["Metronidazole", "Azelaic Acid"],
    "Vitiligo": ["Topical Steroids", "Phototherapy"],
    "Warts": ["Salicylic Acid", "Cryotherapy"],
    "Normal": ["No treatment required"]
}


In [ ]:
upload = FileUpload(accept='image/*', multiple=False)
btn = Button(description="Predict", button_style="success")
out = Output()


In [ ]:
def predict_image(b):
    out.clear_output()

    if len(upload.value) == 0:
        with out:
            print("⚠ Please upload an image first")
        return

    image_bytes = list(upload.value.values())[0]['content']
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    img_resized = img.resize((224,224))

    img_arr = np.array(img_resized) / 255.0
    img_arr = np.expand_dims(img_arr, axis=0)

    preds = model.predict(img_arr)[0]
    idx = np.argmax(preds)
    confidence = preds[idx] * 100
    disease = class_names[idx]

    with out:
        display(img)
        print("🩺 Disease:", disease)
        print("📊 Confidence:", round(confidence,2), "%")
        print("\n💊 Recommended Creams:")
        for c in recommendations.get(disease, ["Consult Dermatologist"]):
            print("•", c)
        print("\n⚠ This is not a medical diagnosis.")


In [ ]:
btn.on_click(predict_image)


In [ ]:
VBox([upload, btn, out])


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\WhatsApp Image 2025-12-13 at 10.38.13 PM.jpeg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
predicted_index = np.argmax(pred)
confidence = pred[0][predicted_index] * 100
predicted_class = class_names[predicted_index]

# Show Image + Prediction
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({confidence:.2f}%)")
plt.show()

# Print details
print("🧬 Predicted Disease:", predicted_class)
print(f"📊 Confidence: {confidence:.2f} %")
print("\n💊 Recommended Creams / Treatment:")
for cream in recommendations.get(predicted_class, ["Consult Dermatologist"]):
 print("•", cream)

In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\WhatsApp Image 2025-12-13 at 11.03.21 PM.jpeg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
predicted_index = np.argmax(pred)
confidence = pred[0][predicted_index] * 100
predicted_class = class_names[predicted_index]

# Show Image + Prediction
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({confidence:.2f}%)")
plt.show()

# Print details
print("🧬 Predicted Disease:", predicted_class)
print(f"📊 Confidence: {confidence:.2f} %")
print("\n💊 Recommended Creams / Treatment:")
for cream in recommendations.get(predicted_class, ["Consult Dermatologist"]):
 print("•", cream)

In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\WhatsApp Image 2025-12-13 at 2.53.27 PM.jpeg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
predicted_index = np.argmax(pred)
confidence = pred[0][predicted_index] * 100
predicted_class = class_names[predicted_index]

# Show Image + Prediction
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({confidence:.2f}%)")
plt.show()

# Print details
print("🧬 Predicted Disease:", predicted_class)
print(f"📊 Confidence: {confidence:.2f} %")
print("\n💊 Recommended Creams / Treatment:")
for cream in recommendations.get(predicted_class, ["Consult Dermatologist"]):
 print("•", cream)

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [ ]:
MODEL_PATH = r"C:\Users\user\Desktop\Skin Detector\archive\models\skin_detector_final.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("✅ Model loaded")


In [ ]:
class_names = [
    "Acne",
    "Eczema",
    "Normal",
    "Psoriasis",
    "Rosacea",
    "Vitiligo",
    "Warts"
]


In [ ]:
recommendations = {
    "Acne": [
        "Benzoyl Peroxide 2.5%",
        "Salicylic Acid Cleanser",
        "Adapalene Gel"
    ],
    "Eczema": [
        "Hydrocortisone Cream",
        "Moisturizers (CeraVe, Cetaphil)",
        "Avoid hot showers"
    ],
    "Normal": [
        "No treatment required",
        "Use sunscreen daily"
    ],
    "Psoriasis": [
        "Coal Tar Cream",
        "Vitamin D analogues",
        "Consult dermatologist"
    ],
    "Rosacea": [
        "Metronidazole Gel",
        "Azelaic Acid",
        "Avoid triggers"
    ],
    "Vitiligo": [
        "Topical Corticosteroids",
        "Phototherapy",
        "Consult dermatologist"
    ],
    "Warts": [
        "Salicylic Acid",
        "Cryotherapy",
        "Consult dermatologist"
    ]
}


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\1800ss_getty_rf_scratching_arm.webp"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
idx = np.argmax(pred)
confidence = pred[0][idx] * 100
predicted_class = class_names[idx]

# Show output
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({confidence:.2f}%)")
plt.show()

print("🧬 Predicted Disease:", predicted_class)
print(f"📊 Confidence: {confidence:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\eczema\1.jpeg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
idx = np.argmax(pred)
confidence = pred[0][idx] * 100
predicted_class = class_names[idx]

# Show output
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({confidence:.2f}%)")
plt.show()

print("🧬 Predicted Disease:", predicted_class)
print(f"📊 Confidence: {confidence:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\eczema\2.webp"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
idx = np.argmax(pred)
confidence = pred[0][idx] * 100
predicted_class = class_names[idx]

# Show output
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({confidence:.2f}%)")
plt.show()

print("🧬 Predicted Disease:", predicted_class)
print(f"📊 Confidence: {confidence:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\eczema\4.jpg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\eczema\5.jpeg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Acne\1.jpg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Acne\2.png"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Acne\3.jpg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Acne\4.jpeg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Acne\5.jpg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Warts\2.jpeg"

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Psorasis\3.jpg"
# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Psorasis\4.jpg"
# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Psorasis\4.jpg"
# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Psorasis\5.jpeg"
# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Vitiligo\1.jpg"
# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Vitiligo\2.jpg"
# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Vitiligo\3.jpeg"
# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [4]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [5]:
MODEL_PATH = r"C:\Users\user\Desktop\Skin Detector\archive\models\skin_detector_final.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("✅ Model loaded")


✅ Model loaded


In [8]:
class_names = [
    "Acne",
    "Eczema",
    "Psoriasis",
    "Rosacea",
    "Vitiligo",
    "Warts"
]


In [9]:
recommendations = {
    "Acne": [
        "Benzoyl Peroxide 2.5%",
        "Salicylic Acid Cleanser",
        "Adapalene Gel"
    ],
    "Eczema": [
        "Hydrocortisone Cream",
        "Moisturizers (CeraVe, Cetaphil)",
        "Avoid hot showers"
    ],
    "Psoriasis": [
        "Coal Tar Cream",
        "Vitamin D analogues",
        "Consult dermatologist"
    ],
    "Rosacea": [
        "Metronidazole Gel",
        "Azelaic Acid",
        "Avoid triggers"
    ],
    "Vitiligo": [
        "Topical Corticosteroids",
        "Phototherapy",
        "Consult dermatologist"
    ],
    "Warts": [
        "Salicylic Acid",
        "Cryotherapy",
        "Consult dermatologist"
    ]
}


In [13]:
IMG_PATH = IMG_PATH = r"C:\Users\user\Downloads\diseases\Warts\2.jpg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
idx = np.argmax(pred)
confidence = pred[0][idx] * 100
predicted_class = class_names[idx]

# Show output
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({confidence:.2f}%)")
plt.show()

print("🧬 Predicted Disease:", predicted_class)
print(f"📊 Confidence: {confidence:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


In [14]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Acne\4.jpeg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


In [3]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\eczema\5.jpeg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


NameError: name 'cv2' is not defined

In [ ]:
IMG_PATH = r"C:\Users\user\Downloads\diseases\Acne\5.jpg"
IMG_SIZE = 224

# Read image
img = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Preprocess
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)[0]

# Disease percentage
disease_percentages = pred * 100
predicted_index = np.argmax(pred)
predicted_class = class_names[predicted_index]
predicted_percent = disease_percentages[predicted_index]

# ================= IMAGE OUTPUT =================
plt.figure(figsize=(5,6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(f"{predicted_class} ({predicted_percent:.2f}%)")
plt.show()

# ================= TEXT OUTPUT =================
print("🧬 Detected Disease:", predicted_class)
print(f"🧪 Disease Percentage: {predicted_percent:.2f} %")

print("\n💊 Recommended Treatment:")
for c in recommendations.get(predicted_class, ["Consult Dermatologist"]):
    print("•", c)


# ================= BAR CHART =================
plt.figure(figsize=(8,5))
plt.bar(class_names, disease_percentages)
plt.xticks(rotation=45)
plt.ylabel("Disease Percentage (%)")
plt.title("Disease Bar Chart")
plt.show()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30

TRAIN_DIR = r"C:\path\to\dataset_6cls\train"
VAL_DIR   = r"C:\path\to\dataset_6cls\val"
TEST_DIR  = r"C:\path\to\dataset_6cls\test"


In [2]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    brightness_range=[0.8,1.2],
    horizontal_flip=True
)

val_test_gen = ImageDataGenerator(rescale=1./255)


NameError: name 'ImageDataGenerator' is not defined

In [1]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [1]:
model.save("skin_detector_final.keras")

NameError: name 'model' is not defined

In [15]:
import tensorflow as tf
from tensorflow.keras.layers import InputLayer

class FixedInputLayer(InputLayer):
    def __init__(self, *args, **kwargs):
        if 'batch_shape' in kwargs:
            kwargs['batch_input_shape'] = kwargs.pop('batch_shape')
        super().__init__(*args, **kwargs)

# Model ka path dein
MODEL_PATH = r"C:\Users\user\Desktop\Skin Detector\archive\models\skin_detector_final.keras"

# Model load karein
model = tf.keras.models.load_model(MODEL_PATH, compile=False, custom_objects={'InputLayer': FixedInputLayer})
print("Model kamyabi se load ho gaya!")

Model kamyabi se load ho gaya!


In [21]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Apni test image ka path yahan dein
IMG_PATH = r"C:\Downloads\diseases\Warts\2.jpg"
IMG_SIZE = 224

class_names = ["Acne", "Eczema", "Psoriasis", "Rosacea", "Vitiligo", "Warts"] # Agar 6 classes hain

# Image read aur preprocess karein
img = cv2.imread(IMG_PATH)
if img is not None:
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_array = img_resized / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Prediction karein
    pred = model.predict(img_array)[0]
    idx = np.argmax(pred)
    confidence = pred[idx] * 100
    predicted_class = class_names[idx]

    # Result show karein
    plt.figure(figsize=(5, 5))
    plt.imshow(img_rgb)
    plt.axis("off")
    plt.title(f"{predicted_class} ({confidence:.2f}%)")
    plt.show()

    print(f"Predicted Disease: {predicted_class}")
    print(f"Confidence: {confidence:.2f} %")
else:
    print("Image load nahi ho saki, path check karein!")

Image load nahi ho saki, path check karein!
